# Legacy Lacunae Analysis Notebook

This notebook now uses the installed `lacunae_analysis` package instead of redefining the workflow inline.
For the full walkthrough, open `notebooks/lacunae_analysis_walkthrough.ipynb`.


In [ ]:
%matplotlib inline

from pathlib import Path

import pandas as pd

from lacunae_analysis.io import load_aim_as_density
from lacunae_analysis.metrics import analyze_lacuna_density
from lacunae_analysis.models import DensityFilterSettings, ScanInput, ThresholdSettings
from lacunae_analysis.pipeline import run_single_scan, run_single_scan_job
from lacunae_analysis.segmentation import plot_lacuna_segmentation, segment_lacunae
from lacunae_analysis.thresholding import compute_threshold


## Configure a Scan

Update the paths below before running the notebook.


In [ ]:
scan_path = Path("path/to/scan.aim")
output_dir = Path("outputs/notebook-legacy")
output_dir.mkdir(parents=True, exist_ok=True)

if not scan_path.exists():
    raise FileNotFoundError("Update `scan_path` to point at a local .aim scan before running the walkthrough.")


## Explore the Shared API

These cells keep the exploratory threshold, segmentation, and density inspection steps while delegating the implementation to package modules.


In [ ]:
loaded_scan = load_aim_as_density(scan_path)
threshold_results = compute_threshold(loaded_scan)
segmentation_results = segment_lacunae(
    loaded_scan,
    threshold=float(threshold_results["selected_threshold"]),
    bone_sigma=10.0,
    lacuna_sigma=1.2,
)
density_results = analyze_lacuna_density(
    segmentation_results["lacuna_binary_sitk"],
    segmentation_results["bone_mask_sitk"],
)

pd.Series(density_results["summary"])


In [ ]:
plot_lacuna_segmentation(segmentation_results)
density_results["component_table"].head()


In [ ]:
pipeline_results = run_single_scan(
    ScanInput(image_path=scan_path, output_dir=output_dir),
    threshold_settings=ThresholdSettings(),
    density_filter_settings=DensityFilterSettings(),
)
pd.Series(pipeline_results["summary"])


In [ ]:
job_results = run_single_scan_job(scan_path, config={}, output_dir=output_dir)
sorted(output_dir.iterdir())
